In [ ]:
!pip install -q torchinfo

# Transfer Learning con PyTorch

Entrenar CNNs desde cero requiere grandes cantidades de datos y tiempo de cómputo. Gran parte de ese esfuerzo se invierte en aprender filtros de bajo nivel que son útiles para cualquier tarea de visión. Surge entonces una pregunta natural: ¿podemos reutilizar una red entrenada en un dataset y adaptarla a una tarea diferente sin repetir el proceso completo?

Eso es precisamente **transfer learning**: transferimos el conocimiento de un modelo preentrenado (generalmente en **ImageNet**) a nuestra tarea específica. Las capas convolucionales ya saben extraer bordes, texturas y formas genéricas; nosotros solo necesitamos adaptar la cabeza de clasificación.

En este notebook exploraremos tres estrategias:
1. Extracción de features con el backbone congelado
2. Reemplazar el clasificador final y entrenar solo esa parte
3. Fine-tuning: descongelar gradualmente el backbone


## 0. Imports y configuración


### Configuración del entorno dual local/Colab

El notebook se **escribe en local** (sin GPU ni datos) y se **ejecuta en Google Colab** (con GPU). Para que corra igual en ambos entornos, la celda de configuración:

- Define `device` como `'cuda'` si hay GPU disponible y `'cpu'` en caso contrario, de modo que el mismo código aproveche la GPU en Colab y siga siendo ejecutable en CPU.
- Fija la semilla de reproducibilidad con `torch.manual_seed(42)`, para que los procesos aleatorios (inicialización de pesos, barajado de datos) sean repetibles entre ejecuciones.
- Define las rutas **relativas y configurables** `DATA_ROOT` y `CKPT_DIR` (en lugar de rutas absolutas), y crea esas carpetas con `os.makedirs(..., exist_ok=True)`. Así, tanto los datos como los checkpoints `.pth` se ubican de forma consistente en local y en Colab.

Estas variables se definen **antes de su primer uso** para que las celdas posteriores construyan sus rutas a partir de `DATA_ROOT`/`CKPT_DIR`.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchinfo import summary
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import glob

# Dispositivo de cómputo: GPU si está disponible (p. ej. en Colab), si no CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando device: {device}')

# Semilla de reproducibilidad: fija el generador de PyTorch para que los
# resultados aleatorios (inicializaciones, barajados) sean repetibles.
torch.manual_seed(42)

# Rutas relativas configurables para que el notebook corra igual en local y en
# Colab, sin literales absolutos. DATA_ROOT: raíz de datos; CKPT_DIR: checkpoints.
DATA_ROOT = 'data'
CKPT_DIR = 'data'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'DATA_ROOT: {DATA_ROOT} | CKPT_DIR: {CKPT_DIR}')


## 1. Funciones auxiliares

Definimos aquí las utilidades que usaremos a lo largo del notebook: verificación del dataset, visualización de muestras, y el loop de entrenamiento con logging a TensorBoard.


In [ ]:
def check_image_dir(pattern):
    """Elimina imágenes corruptas que coincidan con el patrón glob."""
    removed = 0
    for fname in glob.glob(pattern):
        try:
            img = Image.open(fname)
            img.verify()
        except Exception:
            print(f'Eliminando archivo corrupto: {fname}')
            os.remove(fname)
            removed += 1
    print(f'{pattern}: {removed} archivos eliminados')


def display_dataset(dataset, n=10):
    """Muestra n imágenes aleatorias del dataset con su etiqueta."""
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(2, n // 2, figsize=(15, 5))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for ax, idx in zip(axes.flat, indices):
        img, label = dataset[idx]
        img = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        title = dataset.classes[label] if hasattr(dataset, 'classes') else str(label)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = loss_fn(out, y)
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return total_loss / total, correct / total


def train(
    model, train_loader, test_loader,
    loss_fn=None, optimizer=None,
    epochs=5, lr=1e-3, writer=None, tag=''
):
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()
    if optimizer is None:
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr
        )
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, loss_fn)
        va_loss, va_acc = eval_epoch(model, test_loader, loss_fn)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        if writer:
            writer.add_scalars(f'{tag}/Loss', {'train': tr_loss, 'val': va_loss}, epoch)
            writer.add_scalars(f'{tag}/Acc',  {'train': tr_acc,  'val': va_acc},  epoch)
        print(f'Epoch {epoch:02d}/{epochs} | '
              f'train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.4f}')
    return history


def plot_results(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='train')
    ax1.plot(history['val_loss'],   label='val')
    ax1.set_title('Loss'); ax1.legend()
    ax2.plot(history['train_acc'], label='train')
    ax2.plot(history['val_acc'],   label='val')
    ax2.set_title('Accuracy'); ax2.legend()
    plt.tight_layout()
    plt.show()

print('Funciones auxiliares definidas ✓')


## 2. Dataset: Cats vs. Dogs

Usaremos el [Kaggle Cats vs. Dogs Dataset](https://www.kaggle.com/c/dogs-vs-cats). Es un conjunto clásico de ~25 000 imágenes de gatos y perros, ideal para experimentar con transfer learning porque el dominio (fotografías naturales) coincide con ImageNet.


### Fuente del dataset

El enunciado del laboratorio y el notebook base indicaban fuentes de datos distintas:

- El **enunciado** (Actividad 1) exige usar el dataset de **Kaggle `kunalgupta2616/dog-vs-cat-images-data`**.
- El **notebook base** descargaba el ZIP de Microsoft (`kagglecatsanddogs_5340.zip`) con `wget` y asumía la estructura de carpetas `data/PetImages/Cat` y `data/PetImages/Dog`.

Usamos la fuente indicada por el enunciado. Por eso reemplazamos la descarga `wget` por una descarga vía la **CLI de Kaggle** del dataset `kunalgupta2616/dog-vs-cat-images-data`. Como la estructura real de carpetas de ese dataset de Kaggle **no se asume** aquí, los nombres de carpetas/clases y la raíz de `ImageFolder` se verifican **por código** en la celda de verificación posterior (no se incrustan literales como `PetImages/Cat`).

> **Nota de credenciales (Colab):** la descarga obtiene las credenciales de Kaggle desde los **Secrets de Colab** (`KAGGLE_USERNAME` y `KAGGLE_KEY`), sin incrustarlas en el código. Configúralos en el panel de Secrets (icono de la llave 🔑) antes de ejecutar.


In [ ]:
# Descarga del dataset desde Kaggle vía su CLI (reemplaza a wget).
# Fuente = Kaggle 'kunalgupta2616/dog-vs-cat-images-data' (no el ZIP de Microsoft).
import os

KAGGLE_DATASET = 'kunalgupta2616/dog-vs-cat-images-data'

# 1) Credenciales desde los Secrets de Colab (no se incrustan en el código).
#    En Colab: panel de Secrets 🔑 con KAGGLE_USERNAME y KAGGLE_KEY.
credenciales_ok = False
try:
    from google.colab import userdata  # type: ignore
    _kaggle_user = userdata.get('KAGGLE_USERNAME')
    _kaggle_key = userdata.get('KAGGLE_KEY')
    if _kaggle_user and _kaggle_key:
        os.environ['KAGGLE_USERNAME'] = _kaggle_user
        os.environ['KAGGLE_KEY'] = _kaggle_key
        credenciales_ok = True
    else:
        print('[AVISO] Faltan credenciales de Kaggle en Secrets de Colab.')
        print('        Configura KAGGLE_USERNAME y KAGGLE_KEY en el panel de Secrets (🔑).')
except ImportError:
    # Fuera de Colab (p. ej. ejecución local): se usan las credenciales del entorno
    # si existen (variables de entorno o ~/.kaggle/kaggle.json). No se incrustan aquí.
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        credenciales_ok = True
    elif os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        credenciales_ok = True
    else:
        print('[AVISO] No se detectaron credenciales de Kaggle.')
        print('        En Colab: configura KAGGLE_USERNAME y KAGGLE_KEY en Secrets (🔑).')
        print('        En local: exporta esas variables o coloca ~/.kaggle/kaggle.json.')

# 2) Descarga vía CLI de Kaggle hacia DATA_ROOT (descomprimida con --unzip).
if credenciales_ok:
    !kaggle datasets download -d {KAGGLE_DATASET} -p "{DATA_ROOT}" --unzip
    print(f'Descarga solicitada a Kaggle: {KAGGLE_DATASET} -> {DATA_ROOT}')
else:
    print('[AVISO] Descarga omitida: configura las credenciales de Kaggle y re-ejecuta esta celda.')


### Verificación de la estructura de carpetas del dataset

No asumimos los nombres de carpetas/clases del dataset de Kaggle (eso sería inventar literales como `PetImages/Cat`). En su lugar, **inspeccionamos por código** el árbol descomprimido bajo `DATA_ROOT` y determinamos **automáticamente** la raíz adecuada para `ImageFolder`.

`ImageFolder` espera una carpeta cuyos **hijos directos** sean subdirectorios de clase (una subcarpeta por clase, con imágenes dentro). La función `encontrar_raiz_imagefolder(base)`:

1. Recorre el árbol de carpetas (`os.walk`) e **imprime el árbol** encontrado para que quede constancia de la estructura real.
2. Busca la carpeta cuyos subdirectorios directos contienen imágenes (es decir, la raíz válida para `ImageFolder`), prefiriendo el caso típico de **2 clases** (gatos vs. perros).
3. Si **no** puede determinar una raíz única por código, imprime el árbol inspeccionado y deja `IMAGEFOLDER_ROOT` con un valor provisional, a confirmar manualmente en Colab.

> **Nota (local vs. Colab):** en local las carpetas de datos están vacías a propósito, por lo que esta celda imprimirá que no hay estructura y dejará `IMAGEFOLDER_ROOT` con un valor provisional. Al ejecutar en Colab (con el dataset ya descargado) la raíz se determinará por código.


In [ ]:
# Inspecciona por código la estructura real descomprimida bajo DATA_ROOT y
# determina la raíz adecuada para ImageFolder, sin asumir literales.
import os

# Extensiones de imagen reconocidas por torchvision.datasets.ImageFolder.
EXTENSIONES_IMG = ('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')


def imprimir_arbol(base, max_entradas=12):
    """Imprime el árbol de carpetas bajo `base` (recortando listados largos de archivos)."""
    print(f'Árbol de directorios bajo: {base}')
    if not os.path.isdir(base):
        print('  (la ruta no existe o no es un directorio)')
        return
    for dirpath, dirnames, filenames in os.walk(base):
        nivel = dirpath.replace(base, '').count(os.sep)
        sangria = '  ' * nivel
        print(f'{sangria}{os.path.basename(dirpath) or dirpath}/')
        subsangria = '  ' * (nivel + 1)
        imgs = [f for f in filenames if f.lower().endswith(EXTENSIONES_IMG)]
        otros = [f for f in filenames if not f.lower().endswith(EXTENSIONES_IMG)]
        for f in (imgs + otros)[:max_entradas]:
            print(f'{subsangria}{f}')
        restantes = len(filenames) - min(len(filenames), max_entradas)
        if restantes > 0:
            print(f'{subsangria}... (+{restantes} archivos más; {len(imgs)} imágenes en total)')


def _tiene_imagenes(carpeta):
    """True si `carpeta` contiene al menos un archivo de imagen (nivel directo)."""
    try:
        for nombre in os.listdir(carpeta):
            if (os.path.isfile(os.path.join(carpeta, nombre))
                    and nombre.lower().endswith(EXTENSIONES_IMG)):
                return True
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return False
    return False


def _subdirectorios(carpeta):
    """Lista de subdirectorios directos de `carpeta` (orden determinista)."""
    try:
        hijos = sorted(os.listdir(carpeta))
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return []
    return [os.path.join(carpeta, h) for h in hijos
            if os.path.isdir(os.path.join(carpeta, h))]


def encontrar_raiz_imagefolder(base):
    """Devuelve la carpeta cuyos hijos directos son subdirectorios de clases con
    imágenes (raíz válida para torchvision.datasets.ImageFolder).

    Estrategia: recorre `base` con os.walk y, para cada carpeta, cuenta cuántos de
    sus subdirectorios directos contienen imágenes. Esos subdirectorios son clases
    candidatas. Se prefiere el candidato con exactamente 2 clases (gatos vs. perros
    es binario); si no, el candidato con más clases. Devuelve None si no hay ninguno
    determinable por código (p. ej. carpeta vacía en local).
    """
    if not os.path.isdir(base):
        return None
    candidatos = []  # (carpeta, nº de subdirectorios-clase con imágenes)
    for dirpath, dirnames, _filenames in os.walk(base):
        subdirs = _subdirectorios(dirpath)
        clases = [d for d in subdirs if _tiene_imagenes(d)]
        if len(clases) >= 2:
            candidatos.append((dirpath, len(clases)))
    if not candidatos:
        return None
    # Preferir exactamente 2 clases (problema binario Cats vs. Dogs); si no, el
    # candidato con mayor nº de clases. Desempate determinista por ruta.
    binarios = [c for c in candidatos if c[1] == 2]
    if binarios:
        return sorted(binarios, key=lambda c: c[0])[0][0]
    return sorted(candidatos, key=lambda c: (-c[1], c[0]))[0][0]


# 1) Imprimir el árbol real descomprimido bajo DATA_ROOT (constancia de estructura).
imprimir_arbol(DATA_ROOT)

# 2) Determinar por código la raíz de ImageFolder.
IMAGEFOLDER_ROOT = encontrar_raiz_imagefolder(DATA_ROOT)

# 3) Si no es determinable por código, dejar un valor provisional a confirmar en Colab.
if IMAGEFOLDER_ROOT is None:
    print('\n[AVISO] No se pudo determinar por código la raíz de ImageFolder.')
    print('        (Esperado en local: las carpetas de datos están vacías.)')
    print('        Revisa el árbol impreso arriba y confirma la raíz en Colab.')
    IMAGEFOLDER_ROOT = DATA_ROOT  # valor provisional: confirmar en Colab
else:
    clases_detectadas = sorted(
        d for d in os.listdir(IMAGEFOLDER_ROOT)
        if os.path.isdir(os.path.join(IMAGEFOLDER_ROOT, d))
    )
    print(f'\nRaíz de ImageFolder determinada por código: {IMAGEFOLDER_ROOT}')
    print(f'Clases detectadas (por código): {clases_detectadas}')

print(f'\nIMAGEFOLDER_ROOT = {IMAGEFOLDER_ROOT!r}')


### Limpieza de corruptas y construcción de `ImageFolder` sobre la raíz verificada

El dataset contiene algunas imágenes corruptas. Antes de construir el `Dataset` las eliminamos con `check_image_dir`, pero **sin asumir literales** de nombres de carpeta (nada de `data/PetImages/Cat`): iteramos sobre las **clases reales** descubiertas como subdirectorios directos de `IMAGEFOLDER_ROOT`, la raíz que la celda de verificación anterior determinó por código.

A continuación construimos `torchvision.datasets.ImageFolder(IMAGEFOLDER_ROOT, ...)` sobre esa misma raíz verificada y dividimos el dataset de forma **defensiva**: el `random_split` base del notebook asumía `20000` imágenes de entrenamiento, lo que falla si el dataset de Kaggle trae `≤ 20000` imágenes. Calculamos `n_train = 20000 if n > 20000 else int(0.8*n)` y fijamos la semilla con `torch.Generator().manual_seed(42)` para reproducibilidad.

> **Nota (local vs. Colab):** en local las carpetas de datos están vacías, por lo que no hay clases que limpiar y el conteo de imágenes será 0. En Colab, con el dataset ya descargado, se limpiará cada clase y el split usará el tamaño real `n`.

In [ ]:
# Limpieza de imágenes corruptas sobre la raíz verificada por código.
# No usamos literales como 'data/PetImages/Cat'/'.../Dog': iteramos sobre las
# clases reales descubiertas como subdirectorios directos de IMAGEFOLDER_ROOT
# (determinada en la celda de verificación anterior).
clases = sorted(
    d for d in os.listdir(IMAGEFOLDER_ROOT)
    if os.path.isdir(os.path.join(IMAGEFOLDER_ROOT, d))
)
if not clases:
    print(f'[AVISO] No se encontraron subdirectorios de clase bajo {IMAGEFOLDER_ROOT!r}.')
    print('        (Esperado en local: las carpetas de datos están vacías.)')
    print('        En Colab, con el dataset descargado, se limpiará cada clase.')
else:
    print(f'Limpiando imágenes corruptas en clases: {clases}')
    for clase in clases:
        # '*' cubre cualquier extensión de imagen, sin asumir solo .jpg.
        check_image_dir(os.path.join(IMAGEFOLDER_ROOT, clase, '*'))

### Transforms y normalización

Los modelos preentrenados en ImageNet esperan imágenes de **224 × 224** píxeles normalizadas con la media y desviación estándar del dataset de entrenamiento original:

```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

Usar estos valores es fundamental: si los activamos con distribuciones de entrada distintas, los pesos preentrenados producirán representaciones incorrectas.

> **TODO 1:** Completa el pipeline de transforms. Debe redimensionar a 256, recortar al centro 224 × 224, convertir a tensor y aplicar la normalización.


In [ ]:
std_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)

# TODO 1: Define `trans` como un transforms.Compose con los pasos indicados arriba.
# Orden exacto: Resize(256) -> CenterCrop(224) -> ToTensor() -> Normalize (std_normalize).
trans = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    std_normalize
])

# --- no modificar desde aquí ---
assert trans is not None, 'Debes definir el pipeline de transforms'
# Construimos ImageFolder sobre la raíz verificada por código (sin literales de clase).
dataset = torchvision.datasets.ImageFolder(IMAGEFOLDER_ROOT, transform=trans)
print(f'Total de imágenes: {len(dataset)} | Clases: {dataset.classes}')

# Split defensivo: el notebook base asumía 20000 de train, lo que falla si el
# dataset trae <= 20000 imágenes. Ajustamos n_train al tamaño real n.
n = len(dataset)
n_train = 20000 if n > 20000 else int(0.8 * n)
trainset, testset = torch.utils.data.random_split(
    dataset, [n_train, n - n_train],
    generator=torch.Generator().manual_seed(42)
)
print(f'Train: {len(trainset)} | Test: {len(testset)}')
display_dataset(dataset)


### Pregunta (Actividad 1): ¿de dónde salen la media y la desviación estándar de ImageNet y por qué importan con pesos preentrenados?

**Origen de los valores.** Los vectores `mean = [0.485, 0.456, 0.406]` y `std = [0.229, 0.224, 0.225]` (en el orden de canales RGB) **no son arbitrarios**: son las estadísticas calculadas sobre el conjunto de entrenamiento de **ImageNet**. Es decir, se tomaron todas las imágenes de entrenamiento, se escalaron sus píxeles al rango `[0, 1]` (como hace `ToTensor()`) y se calcularon, por canal de color, el promedio (media) y la dispersión (desviación estándar) de los valores de los píxeles. Esos números resumen "cómo luce en promedio" un píxel de ImageNet en cada canal.

**Qué hace la normalización.** `transforms.Normalize(mean, std)` aplica, canal a canal, la transformación `x_norm = (x - mean) / std`. El resultado es que cada canal queda **centrado alrededor de 0** y con una **escala comparable** (varianza cercana a 1). Esta es la misma transformación que se aplicó a las imágenes cuando el modelo (VGG-16, ResNet-18, etc.) fue **entrenado originalmente** en ImageNet.

**Por qué importa al usar pesos preentrenados.** Una red preentrenada aprendió sus filtros y pesos esperando recibir entradas con **esa distribución concreta** (centrada y escalada con la media/std de ImageNet). Los pesos de las primeras capas están calibrados para ese rango de valores de entrada. Si alimentamos la red con imágenes normalizadas de otra forma (o sin normalizar), las activaciones internas quedan **fuera del rango esperado**: los filtros preentrenados responden de manera distinta a la que fueron ajustados, las representaciones intermedias se degradan y, en consecuencia, tanto la inferencia directa como el *transfer learning* (extracción de features o fine-tuning) parten de representaciones de peor calidad. En términos prácticos, respetar `mean`/`std` de ImageNet garantiza que las imágenes de nuestro problema (Cats vs. Dogs) "se vean", desde el punto de vista estadístico, como las que el modelo ya sabe procesar, maximizando la utilidad del conocimiento transferido.

> En resumen: la media/std provienen de las estadísticas del set de entrenamiento de ImageNet, y usarlas es imprescindible porque los pesos preentrenados fueron ajustados sobre entradas normalizadas exactamente así; cambiarlas desalinea la distribución de entrada respecto a la que la red espera.


## 3. Modelos preentrenados

`torchvision.models` incluye decenas de arquitecturas con pesos preentrenados en ImageNet. Cargamos VGG-16 y lo usamos directamente para hacer una inferencia sobre una imagen del dataset:


In [ ]:
vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)
vgg = vgg.to(device)

sample_image, _ = dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)

with torch.no_grad():
    res = vgg(sample_image)

# Verificación por código: VGG-16 preentrenado en ImageNet clasifica en 1000
# clases, por lo que la salida de una sola imagen debe tener shape [1, 1000].
assert res.shape == torch.Size([1, 1000]), f'Shape inesperado: {res.shape}'

print(f'Shape de salida: {res.shape}')  # [1, 1000]
print(f'Clase predicha (índice ImageNet): {res[0].argmax().item()}')


In [ ]:
# Visualizar la imagen
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

img_show = (sample_image.squeeze(0).cpu() * std + mean).clamp(0, 1)
img_show = img_show.permute(1, 2, 0).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(img_show)
plt.title(f'Clase predicha: {res[0].argmax().item()}')
plt.axis('off')
plt.show()

La red devuelve un índice de clase de ImageNet. Podemos descargar el mapeo para ver el nombre de la predicción:


In [ ]:
import json, requests

class_map = json.loads(
    requests.get('https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json').text
)
class_map = {int(k): v for k, v in class_map.items()}

pred_idx = res[0].argmax().item()
print(f'Clase predicha: {class_map[pred_idx]}')


### Arquitectura de VGG-16

Inspeccionemos la arquitectura completa con `torchinfo.summary`. Fíjate en el número total de parámetros y en los tipos de capas que aparecen:

> **Nota sobre `input_size`.** VGG-16 preentrenado en ImageNet espera entradas de **224 × 224** (coherente con el `CenterCrop(224)` del pipeline de transforms). Por eso todas las llamadas a `summary` en el notebook usan `input_size=(1, 3, 224, 224)`.


In [ ]:
# input_size usa 224: VGG-16 espera entradas de 224x224.
# El número total de parámetros NO se afirma en texto: se lee del resumen
# impreso por `summary` al ejecutar (fila 'Total params').
summary(vgg, input_size=(1, 3, 224, 224))


Además de las capas convolucionales y de pooling que ya conoces, observarás capas **Dropout**. Actúan como técnica de **regularización**: durante el entrenamiento desactivan aleatoriamente una fracción de las neuronas de la capa anterior, lo que obliga a la red a aprender representaciones más robustas y distribuidas.

La arquitectura se divide en tres partes:
- `vgg.features` — el extractor convolucional
- `vgg.avgpool` — pooling adaptativo
- `vgg.classifier` — capas densas que mapean 25 088 features → 1 000 clases


### Pregunta (Actividad 2): nº total de parámetros, qué hace Dropout y por qué 1 000 salidas

**1) Número total de parámetros de VGG-16.** No afirmamos aquí una cifra concreta: el número total de parámetros se **obtiene por código** a partir del resumen que imprime `torchinfo.summary(vgg, input_size=(1, 3, 224, 224))` en la celda anterior. Al ejecutar en Colab, hay que leer la fila **`Total params`** de esa salida (y, si interesa, `Trainable params`). VGG-16 es una red con muchos parámetros porque su clasificador denso (dos capas de 4096 neuronas) concentra la mayor parte de ellos; el valor exacto queda determinado por la salida de `summary` y no por una afirmación en texto.

**2) Qué hace Dropout y por qué mejora la generalización.** Las capas `Dropout` que aparecen en `vgg.classifier` son una técnica de **regularización**. Durante el **entrenamiento**, Dropout pone a cero de forma **aleatoria** una fracción `p` de las activaciones de la capa (por defecto `p = 0.5` en VGG-16), reescalando las restantes para mantener la magnitud esperada. Esto tiene dos efectos: (a) impide que la red dependa en exceso de neuronas individuales (coadaptación), forzándola a distribuir el conocimiento en representaciones más robustas; y (b) equivale, de forma aproximada, a entrenar y promediar un **ensamble** de muchas subredes. El resultado es una menor tendencia al **sobreajuste** y, por tanto, mejor **generalización** a datos no vistos. En **inferencia** (`model.eval()`) Dropout se desactiva: usa todas las neuronas, de modo que la predicción es determinista.

**3) Por qué la última capa tiene exactamente 1 000 salidas.** Porque VGG-16 fue **entrenado en ImageNet (ILSVRC)**, cuyo problema de clasificación tiene **1 000 categorías**. La última capa lineal del clasificador produce un **logit por clase**, es decir, 1 000 valores que (tras softmax) representan la probabilidad de cada una de las 1 000 clases de ImageNet. Cuando adaptamos la red a nuestro problema binario Cats vs. Dogs, precisamente **reemplazamos** esa última capa (o todo el clasificador) por una capa con **2 salidas**, una por clase de nuestra tarea.


## 4. Extracción de features

Podemos usar `vgg.features` como extractor fijo: pasamos imágenes, obtenemos tensores de representación, y luego entrenamos un clasificador pequeño encima de esos vectores. Así evitamos reentrenar toda la red.

Primero visualicemos los mapas de activación de la última capa convolucional:


In [ ]:
with torch.no_grad():
    feat = vgg.features(sample_image).cpu()

print(f'Shape del tensor de features: {feat.shape}')  # [1, 512, 7, 7]

plt.figure(figsize=(15, 3))
plt.imshow(feat.squeeze().view(512, -1).T, aspect='auto', cmap='viridis')
plt.title('Feature maps de VGG (512 canales × 49 posiciones espaciales)')
plt.colorbar()
plt.show()


Ahora pre-computamos los feature vectors para un subconjunto de 800 imágenes. Almacenamos los resultados en tensores para poder entrenar el clasificador sin recalcular features en cada epoch.

> **TODO 2:** Completa el loop de extracción. Inicializa `feature_tensor` de shape `[num, 512*7*7]` y `label_tensor` de shape `[num]`, luego llena ambos tensores dentro del bucle.


### Explicación: extracción de features con el backbone congelado

En esta celda recorremos `num = bs * 100 = 800` imágenes y precomputamos sus vectores de features usando únicamente la parte convolucional de VGG-16 (`vgg.features`). La idea es:

- **Inicialización.** Reservamos de antemano los tensores de salida con su shape final: `feature_tensor` de shape `[800, 512*7*7]` (es decir `[800, 25088]`, porque la última capa convolucional de VGG-16 produce mapas de `512` canales de `7x7`) y `label_tensor` de shape `[800]`. Usamos la variable `num` en lugar de literales para que el código quede acoplado al tamaño real del subconjunto.
- **`torch.no_grad()`.** Como solo queremos las activaciones y no vamos a entrenar el backbone, desactivamos el cálculo de gradientes. Esto ahorra memoria y acelera el forward.
- **Aplanado.** `vgg.features(x)` devuelve un tensor `[bs, 512, 7, 7]`; con `.flatten(1)` lo convertimos en `[bs, 512*7*7]`, un vector de features por imagen, y lo movemos a CPU con `.cpu()` para almacenarlo.
- **Almacenamiento por lotes.** Escribimos cada batch en la posición `i:i+n` de los tensores. El tamaño efectivo `n = min(f.shape[0], num - i)` recorta el último batch si fuera parcial, de modo que nunca escribimos más de `num` filas aunque el último lote del `DataLoader` no esté completo.

Al final imprimimos los shapes de ambos tensores para verificar que son `[800, 25088]` y `[800]` respectivamente.

In [ ]:
bs  = 8
num = bs * 100  # 800 imágenes
dl  = torch.utils.data.DataLoader(dataset, batch_size=bs, shuffle=True)

# TODO 2: Inicializa feature_tensor y label_tensor
feature_tensor = torch.zeros(num, 512 * 7 * 7)  # shape [num, 512*7*7]
label_tensor   = torch.zeros(num)               # shape [num]

i = 0
for x, l in dl:
    with torch.no_grad():
        # TODO 2 (cont.): extrae features, aplana y almacena en los tensores
        f = vgg.features(x.to(device)).flatten(1).cpu()  # [bs, 512*7*7]
        n = min(f.shape[0], num - i)  # maneja el último batch parcial
        feature_tensor[i:i + n] = f[:n]
        label_tensor[i:i + n]   = l[:n]
    i += bs
    print('.', end='')
    if i >= num:
        break

print(f'\nfeature_tensor: {feature_tensor.shape} | label_tensor: {label_tensor.shape}')


## 5. Clasificador lineal sobre features fijas

Con los feature vectors pre-computados, entrenamos un clasificador de una sola capa lineal. Esto es extremadamente rápido porque no hay que hacer backprop a través de la CNN.

La función de pérdida estándar para clasificación multiclase en PyTorch es `CrossEntropyLoss`, que internamente aplica softmax. Por eso la red **no debe** incluir una capa de activación final — la red devuelve logits crudos.

> **TODO 3:** Define `net` como un `nn.Sequential` con una sola capa `Linear` de `512*7*7` entradas y `2` salidas.


In [ ]:
vgg_dataset = torch.utils.data.TensorDataset(
    feature_tensor,
    label_tensor.to(torch.long)
)
train_ds, test_ds = torch.utils.data.random_split(vgg_dataset, [700, 100])
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=32)

# TODO 3: Define net — una Linear(512*7*7, 2) es suficiente con CrossEntropyLoss.
# Una sola capa lineal basta: produce 2 logits crudos. NO se añade LogSoftmax
# porque la pérdida (nn.CrossEntropyLoss, por defecto en train) ya aplica
# LogSoftmax internamente; añadirlo lo aplicaría dos veces (ver pregunta abajo).
net = nn.Linear(512 * 7 * 7, 2)
net = net.to(device)

writer  = SummaryWriter('runs/linear_classifier')
history = train(net, train_loader, test_loader, epochs=5, writer=writer, tag='linear')
writer.close()
plot_results(history)


### Pregunta (Actividad 3): ¿cuál es la diferencia entre `CrossEntropyLoss` y `NLLLoss + LogSoftmax`, y por qué combinarlos es un error?

**Qué calcula cada pieza.** En clasificación, el modelo produce un vector de **logits** (puntuaciones crudas, sin acotar). Para convertirlos en una pérdida de clasificación hay dos piezas conceptuales:

- **`LogSoftmax`** es una capa de activación: toma los logits y devuelve el **logaritmo de las probabilidades** softmax, es decir `log(softmax(x))`. Numéricamente es más estable que calcular `softmax` y luego `log` por separado.
- **`NLLLoss`** (Negative Log-Likelihood) es una pérdida que **espera recibir log-probabilidades** ya calculadas: simplemente toma el valor correspondiente a la clase verdadera y le cambia el signo (promediando sobre el batch). Por sí sola **no** aplica ninguna softmax ni logaritmo.

**La diferencia.** `nn.CrossEntropyLoss` **no** es una alternativa distinta: es exactamente la **composición de ambas** en un solo módulo. Internamente aplica `LogSoftmax` a los logits y luego `NLLLoss`. Por eso:

- `CrossEntropyLoss(logits, y)` ≡ `NLLLoss(LogSoftmax(logits), y)`.
- Si usas `CrossEntropyLoss`, tu red debe devolver **logits crudos** (sin activación final).
- Si usas `NLLLoss`, tu red **sí** debe terminar en una capa `LogSoftmax` (porque `NLLLoss` espera log-probabilidades, no logits).

**Por qué combinarlos (CrossEntropyLoss + LogSoftmax) es un error.** Si la red ya termina en `LogSoftmax` y además usas `CrossEntropyLoss`, el `LogSoftmax` se aplica **dos veces**: una en la última capa de la red y otra dentro de `CrossEntropyLoss`. Esa doble aplicación distorsiona la señal (se aplica softmax sobre valores que ya son log-probabilidades, no logits), aplasta las diferencias entre clases, produce **gradientes incorrectos/atenuados** y, en la práctica, un entrenamiento peor o que no converge bien. El error es equivalente, en el otro sentido, a usar `NLLLoss` con una red que devuelve logits crudos (sin `LogSoftmax`): entonces falta la activación que `NLLLoss` da por supuesta.

**Decisión en este notebook.** Como entrenamos con `nn.CrossEntropyLoss` (el valor por defecto de la función `train`), definimos `net = nn.Linear(512*7*7, 2)` **sin** ninguna activación final: la red devuelve 2 logits crudos y la pérdida se encarga del `LogSoftmax`. Añadir un `LogSoftmax` aquí sería precisamente el error de doble aplicación descrito.

> **Regla práctica:** `CrossEntropyLoss` ⟶ red con logits crudos (sin activación final). `NLLLoss` ⟶ red terminada en `LogSoftmax`. Nunca `CrossEntropyLoss` **y** `LogSoftmax` juntos.


## 6. Transfer learning — Reemplazar el clasificador

En lugar de pre-computar features, podemos conectar directamente nuestra cabeza de clasificación al backbone de VGG y entrenar el modelo end-to-end. Para ello:

1. **Reemplazamos `vgg.classifier`** por una sola capa `Linear(25088, 2)`.
2. **Congelamos `vgg.features`** poniendo `requires_grad = False` en todos sus parámetros.    Esto es importante: si entrenamos end-to-end desde el principio, los errores grandes    del clasificador aleatorio destruirían los pesos preentrenados del extractor.

> **TODO 4:** Carga VGG-16 con pesos preentrenados, reemplaza el clasificador y congela las capas convolucionales.


In [ ]:
# TODO 4a: Carga VGG-16 limpio con pesos preentrenados (API moderna de pesos).
vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)
vgg = vgg.to(device)

# TODO 4b: Reemplaza vgg.classifier por una sola capa Linear(25088, 2).
# 25088 = 512*7*7 son las features aplanadas que entrega vgg.features (vía avgpool).
vgg.classifier = nn.Linear(25088, 2).to(device)

# TODO 4c: Congela los parámetros de vgg.features (requires_grad = False) para que
# el entrenamiento solo actualice el nuevo clasificador y no destruya los pesos
# preentrenados del extractor convolucional.
for p in vgg.features.parameters():
    p.requires_grad = False

# Conteo de parámetros entrenables POR CÓDIGO (no se afirma el valor en texto).
# Tras congelar features, solo el clasificador lineal queda entrenable.
entrenables = sum(p.numel() for p in vgg.parameters() if p.requires_grad)
print(f'Parámetros entrenables (calculados por código): {entrenables}')

# Verifica: solo el clasificador debe tener parámetros entrenables.
# input_size usa 224: VGG-16 espera entradas de 224x224.
summary(vgg, input_size=(1, 3, 224, 224))


### Pregunta (Actividad 4): ¿por qué es importante congelar las capas convolucionales antes de entrenar el nuevo clasificador?

Cuando reemplazamos `vgg.classifier` por una capa `Linear(25088, 2)` nueva, esa capa arranca con **pesos aleatorios**. En las primeras iteraciones sus predicciones son prácticamente aleatorias, por lo que la pérdida es alta y los **gradientes que genera son grandes**. Si en ese momento las capas convolucionales (`vgg.features`) estuvieran entrenables, esos gradientes grandes se **retropropagarían** hacia el extractor y modificarían de golpe unos pesos preentrenados que ya son valiosos: la red tardó mucho (y muchísimos datos de ImageNet) en aprender a detectar bordes, texturas y formas genéricas. Dejar `requires_grad = True` en el backbone desde el inicio equivale a **arrastrar un buen extractor hacia el ruido** del clasificador recién inicializado, degradando el conocimiento transferido (fenómeno de *destruir* los pesos preentrenados).

Congelar `vgg.features` con `requires_grad = False` evita ese problema por varias razones:

- **Protege el conocimiento preentrenado.** Al no calcular ni aplicar gradientes sobre el backbone, sus filtros permanecen intactos mientras el clasificador aún es inestable. El extractor sigue produciendo representaciones de alta calidad desde la primera época.
- **Estabiliza y acelera el entrenamiento.** Solo se optimiza una capa lineal pequeña (muy pocos parámetros entrenables frente a los millones del backbone), lo que hace que el entrenamiento sea más rápido, consuma menos memoria y converja de forma más estable.
- **Reduce el riesgo de sobreajuste.** Con pocos parámetros entrenables y un dataset limitado, la cabeza aprende la tarea (Cats vs. Dogs) sin que la red entera se memorice los datos.

La estrategia habitual es, por tanto, **en dos etapas**: primero se entrena solo la cabeza con el backbone congelado (esta Actividad 4) hasta que el clasificador tenga pesos razonables; después, de forma **opcional**, se **descongela** el backbone y se hace *fine-tuning* con un **learning rate pequeño** (Actividad 5), de modo que ahora sí los gradientes que llegan a las convolucionales son suaves y especializan el extractor a nuestro dominio sin destruir lo aprendido.


La celda anterior **calcula por código** el número de parámetros entrenables (la línea `Parámetros entrenables (calculados por código): ...`) y lo contrasta con la fila `Trainable params` del resumen de `summary`. Tras congelar `vgg.features`, los únicos parámetros entrenables son los de la nueva capa lineal del clasificador; el valor concreto se lee de esa salida al ejecutar, no se afirma aquí en texto.


### Entrenamiento de M2 congelado (1 época) y guardado del checkpoint

Con el clasificador reemplazado y `vgg.features` congelado, entrenamos ahora el modelo **M2 en su etapa congelada** sobre el **dataset completo**. Como solo la cabeza lineal es entrenable, el *backbone* preentrenado permanece intacto y actúa como extractor de features fijo. Pasos de esta celda:

- **DataLoaders.** Reutilizamos los *splits* `trainset`/`testset` definidos en la Sec. 2 (split defensivo con semilla 42) y construimos sus `DataLoader` de entrenamiento y validación.
- **Entrenamiento.** Llamamos a la función base `train` por **1 época** con `nn.CrossEntropyLoss`, registrando las curvas en **TensorBoard** con el **tag `frozen`** (bajo `runs/frozen`), para poder compararlas luego con los demás experimentos. Después dibujamos las curvas con `plot_results` para que el reporte sea autocontenido.
- **Checkpoint.** Guardamos los pesos con `torch.save(vgg.state_dict(), ...)` en una ruta construida con `CKPT_DIR` (sin literales absolutos): `os.path.join(CKPT_DIR, 'cats_dogs_weights.pth')`. Este checkpoint es el que recargará la Actividad 5 (fine-tuning) para descongelar el backbone y continuar el entrenamiento.

> **Nota (local vs. Colab):** este entrenamiento requiere los datos descargados y, en la práctica, **GPU** (es un proceso costoso); está pensado para ejecutarse en Colab. En local, sin datos, esta celda no se ejecuta.


In [ ]:
# Reutilizamos los splits trainset/testset definidos en la Sec. 2 (split
# defensivo con semilla 42). Construimos los DataLoaders de train/test.
train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

# Entrenamiento de 1 época sobre el dataset completo con tag 'frozen' en TensorBoard.
writer = SummaryWriter('runs/frozen')
history_frozen = train(
    vgg, train_loader, test_loader,
    loss_fn=nn.CrossEntropyLoss(),
    epochs=1, lr=1e-3,
    writer=writer, tag='frozen'
)
writer.close()
plot_results(history_frozen)

# Guardamos el checkpoint usando CKPT_DIR (sin literal absoluto), para que la
# Actividad 5 (fine-tuning) pueda recargar estos pesos.
ckpt_path = os.path.join(CKPT_DIR, 'cats_dogs_weights.pth')
torch.save(vgg.state_dict(), ckpt_path)
print(f'Modelo guardado ✓ -> {ckpt_path}')


## 7. Fine-tuning — Descongelar el backbone

Una vez que el clasificador tiene pesos razonables, podemos descongelar las capas convolucionales y continuar entrenando toda la red con un **learning rate más pequeño**. Esto permite que el extractor se especialice en nuestro dominio.

> ⚠️ Si descongelamos desde el principio (sin estabilizar primero el clasificador), los gradientes grandes del clasificador aleatorio pueden degradar los pesos preentrenados.

> **TODO 5:** Descongela los parámetros de `vgg.features` y entrena una epoch adicional con `lr=1e-4`.


### TODO 5 — Fine-tuning de M2 (descongelar `features` y entrenar con `lr=1e-4`)

En esta celda completamos la etapa de *fine-tuning* del modelo **M2**. Los pasos son:

- **Reconstruir la arquitectura antes de cargar el checkpoint.** Creamos `vgg = torchvision.models.vgg16(weights=None)` (sin descargar pesos preentrenados) y reemplazamos su cabeza por `vgg.classifier = nn.Linear(25088, 2)`, igual que en la Actividad 4. Esto es necesario para que las **claves** del `state_dict` guardado coincidan antes de `load_state_dict`.
- **Cargar el checkpoint con `CKPT_DIR`.** Leemos los pesos entrenados en la etapa congelada desde `os.path.join(CKPT_DIR, 'cats_dogs_weights.pth')` (sin rutas absolutas), usando `weights_only=True`. Es exactamente el archivo que guardó la Sec. 6.
- **Descongelar el backbone.** Ponemos `requires_grad = True` en todos los parámetros de `vgg.features`. A diferencia de la etapa congelada (donde solo entrenaba la cabeza), ahora los gradientes también llegan a las capas convolucionales, de modo que el extractor puede **especializarse** en el dominio Cats vs. Dogs. Imprimimos por código el número de parámetros entrenables para dejar constancia (sin afirmar el valor en texto).
- **Entrenar 1 época con `lr=1e-4`.** Usamos un *learning rate* pequeño para no destruir lo aprendido en ImageNet: pasos de actualización suaves que ajustan los filtros preentrenados en lugar de reescribirlos. Reutilizamos la función base `train` y los `DataLoader` de la Sec. 6 (o los reconstruimos de forma defensiva desde `trainset`/`testset`).
- **Logging comparable.** Registramos las curvas en TensorBoard con el **tag `finetuned`** bajo `runs/finetuned`, un directorio **separado** de `runs/frozen`. Así, las curvas de la etapa **congelada** (Sec. 6, tag `frozen`) y la etapa **fine-tuned** (esta celda, tag `finetuned`) quedan como series distintas y comparables en el panel de TensorBoard. Además dibujamos las curvas con `plot_results` para que el reporte sea autocontenido.

> **Nota (local vs. Colab):** este fine-tuning requiere el checkpoint de la Sec. 6, los datos descargados y, en la práctica, **GPU**; está pensado para ejecutarse en Colab. En local, sin datos ni checkpoint, esta celda no se ejecuta.


In [ ]:
# Reconstruimos primero la arquitectura (vgg16 sin pesos + classifier Linear(25088, 2))
# para que las claves coincidan antes de cargar el state_dict guardado en la Sec. 6.
vgg = torchvision.models.vgg16(weights=None)
vgg.classifier = nn.Linear(25088, 2)
vgg = vgg.to(device)

# Cargamos el checkpoint usando CKPT_DIR (sin literal absoluto): es el mismo
# archivo que guardó la Actividad 4 (etapa congelada).
ckpt_path = os.path.join(CKPT_DIR, 'cats_dogs_weights.pth')
vgg.load_state_dict(torch.load(ckpt_path, weights_only=True))
print(f'Checkpoint cargado ✓ <- {ckpt_path}')

# TODO 5: Descongela vgg.features para permitir el fine-tuning del backbone.
for p in vgg.features.parameters():
    p.requires_grad = True
entrenables_ft = sum(p.numel() for p in vgg.parameters() if p.requires_grad)
print(f'Parámetros entrenables tras descongelar features: {entrenables_ft:,}')

# Reutilizamos los DataLoaders de la Sec. 6 si existen; si no, los reconstruimos
# de forma defensiva a partir de los splits trainset/testset de la Sec. 2.
if 'train_loader' not in globals() or 'test_loader' not in globals():
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

# Entrenamos 1 época con lr=1e-4. Usamos el tag 'finetuned' y un directorio
# propio bajo runs/ (runs/finetuned), separado de runs/frozen, para que las
# curvas congelado vs. fine-tuned sean comparables en TensorBoard.
writer = SummaryWriter('runs/finetuned')
history_ft = train(
    vgg, train_loader, test_loader,
    loss_fn=nn.CrossEntropyLoss(),
    epochs=1, lr=1e-4,
    writer=writer, tag='finetuned'
)
writer.close()
plot_results(history_ft)


### Experimento: learning rate scheduler en el fine-tuning (preguntas 3 y 7)

Las preguntas de investigación **3 y 7** del enunciado piden investigar `torch.optim.lr_scheduler.StepLR` y `CosineAnnealingLR`, agregar uno al **loop de fine-tuning** de la Actividad 5 y **comparar las curvas**. Aquí implementamos ese experimento como un **loop de fine-tuning dedicado** con un *scheduler*.

**¿Por qué una celda aparte y no la función base `train`?** La función `train` del notebook no expone un parámetro de *scheduler*. Por eso, en lugar de modificar la utilidad base (que otras celdas reutilizan), escribimos un **loop explícito** que reutiliza las funciones base `train_epoch` y `eval_epoch` y añade el *scheduler*:

- `optimizer = optim.Adam(...)` sobre los parámetros entrenables, con el mismo `lr=1e-4` del fine-tuning base, para que la comparación sea **justa** (misma arquitectura, mismo punto de partida, mismo `lr` inicial).
- `scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_SCHED)` (alternativa documentada: `StepLR(optimizer, step_size=1, gamma=0.5)`). El *scheduler* **reduce el learning rate a lo largo de las épocas**: `CosineAnnealingLR` lo baja siguiendo una curva coseno hasta ~0, mientras que `StepLR` lo multiplica por `gamma` cada `step_size` épocas.
- **Dónde va cada `step()`.** `optimizer.step()` se ejecuta **por batch** (dentro de `train_epoch`, que ya lo hace); `scheduler.step()` se ejecuta **por época** (al final de cada época, en este loop). Este es el patrón correcto para los *schedulers* por época.

**Qué comparar.** Reconstruimos el mismo modelo M2 desde el checkpoint de la Sec. 6 (`vgg16(weights=None)` + `classifier = nn.Linear(25088, 2)` + `load_state_dict`) y descongelamos `features`, de modo que la **única diferencia** frente al fine-tuning base (tag `finetuned`) sea el *scheduler*. Registramos las curvas en TensorBoard con un **tag distinto** (`finetuned_sched`, bajo `runs/finetuned_sched`) para poder superponerlas con las de `finetuned` en el panel, y además las dibujamos con `plot_results` dentro del notebook. Usamos **unas pocas épocas** (`EPOCHS_SCHED`) para que las curvas sean comparables con el fine-tuning base sin alargar el entrenamiento.

> **Nota (local vs. Colab):** este experimento requiere el checkpoint de la Sec. 6, los datos descargados y, en la práctica, **GPU**; está pensado para ejecutarse en Colab. En local, sin datos ni checkpoint, esta celda no se ejecuta.


In [ ]:
# Experimento (preguntas 3 y 7): fine-tuning con un learning rate scheduler.
# Loop explícito que reutiliza train_epoch/eval_epoch (la función base `train` no
# expone scheduler) y añade un scheduler por época. Tag distinto: 'finetuned_sched'.
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR

# Nº pequeño de épocas para que las curvas sean comparables con el fine-tuning base.
EPOCHS_SCHED = 3

# 1) Reconstruir M2 desde el checkpoint de la Sec. 6 (misma arquitectura y punto de
#    partida que el fine-tuning base, para una comparación justa).
ckpt_path = os.path.join(CKPT_DIR, 'cats_dogs_weights.pth')
vgg_sched = torchvision.models.vgg16(weights=None)
vgg_sched.classifier = nn.Linear(25088, 2)
vgg_sched.load_state_dict(torch.load(ckpt_path, weights_only=True))
vgg_sched = vgg_sched.to(device)

# 2) Descongelar el backbone (igual que el fine-tuning base): única diferencia = scheduler.
for p in vgg_sched.features.parameters():
    p.requires_grad = True

# 3) Reutilizar los DataLoaders si existen; si no, reconstruirlos de forma defensiva.
if 'train_loader' not in globals() or 'test_loader' not in globals():
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

# 4) Optimizer con el mismo lr=1e-4 del fine-tuning base + scheduler por época.
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, vgg_sched.parameters()), lr=1e-4
)
# CosineAnnealingLR baja el lr siguiendo una curva coseno hasta ~0 a lo largo de
# T_max épocas. Alternativa equivalente (comentada): StepLR cada época.
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_SCHED)
# scheduler = StepLR(optimizer, step_size=1, gamma=0.5)  # alternativa (pregunta 3)

# 5) Loop explícito: optimizer.step() va por batch (dentro de train_epoch);
#    scheduler.step() va por época (aquí, al final de cada época).
writer = SummaryWriter('runs/finetuned_sched')
history_sched = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
for epoch in range(1, EPOCHS_SCHED + 1):
    tr_loss, tr_acc = train_epoch(vgg_sched, train_loader, optimizer, loss_fn)
    va_loss, va_acc = eval_epoch(vgg_sched, test_loader, loss_fn)
    scheduler.step()  # actualización del learning rate POR ÉPOCA
    history_sched['train_loss'].append(tr_loss)
    history_sched['train_acc'].append(tr_acc)
    history_sched['val_loss'].append(va_loss)
    history_sched['val_acc'].append(va_acc)
    lr_actual = optimizer.param_groups[0]['lr']
    writer.add_scalars('finetuned_sched/Loss', {'train': tr_loss, 'val': va_loss}, epoch)
    writer.add_scalars('finetuned_sched/Acc',  {'train': tr_acc,  'val': va_acc},  epoch)
    writer.add_scalar('finetuned_sched/LR', lr_actual, epoch)
    print(f'Epoch {epoch:02d}/{EPOCHS_SCHED} | '
          f'train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
          f'val loss {va_loss:.4f} acc {va_acc:.4f} | lr {lr_actual:.2e}')
writer.close()

# 6) Curvas dibujadas dentro del notebook (reporte autocontenido).
plot_results(history_sched)


#### Conclusión del experimento de scheduler (preguntas 3 y 7)

Compara las curvas del fine-tuning **con** scheduler (`history_sched`, tag `finetuned_sched`) frente al fine-tuning **base** (`history_ft`, tag `finetuned`) en TensorBoard (`%tensorboard --logdir runs`) y con las gráficas de `plot_results` dibujadas arriba. Observa el impacto del *scheduler* en la estabilidad de las curvas de pérdida/accuracy de validación y en la velocidad de convergencia a lo largo de las épocas.

<!-- COMPLETAR tras ejecución -->

_(Respuesta empírica pendiente de ejecución en Colab. Tras ejecutar, describe aquí: si el `scheduler` mejoró o estabilizó la curva de validación respecto al fine-tuning sin scheduler, cómo evolucionó el learning rate — ver `finetuned_sched/LR` en TensorBoard — y si hubo diferencias en la accuracy final de validación.)_


### Experimento: data augmentation (pregunta 2)

La **pregunta de investigación 2** del enunciado pide experimentar con *data augmentation* y
analizar su **impacto en la generalización**. Aquí lo implementamos como un **experimento
autocontenido** (no depende de ResNet ni de la Sec. 8): construimos un pipeline de
**entrenamiento con augmentations** y entrenamos un modelo M2 (VGG-16 con `features`
congeladas + cabeza lineal) registrando sus curvas en TensorBoard con un **tag distinto**
(`aug`, bajo `runs/aug`).

**Qué es *data augmentation* y por qué puede ayudar.** Son transformaciones aleatorias que se
aplican a las imágenes de entrenamiento (voltear, rotar, variar brillo/contraste...). Cada época
el modelo ve variantes ligeramente distintas de las mismas imágenes, lo que **amplía
artificialmente** el conjunto de entrenamiento y suele **reducir el sobreajuste**, mejorando la
generalización a datos no vistos.

**Diseño del experimento (comparación justa):**

- **Transforms de entrenamiento (con augmentation):** `RandomHorizontalFlip`, `RandomRotation`
  y `ColorJitter` aplicados **encima** del pipeline base
  `Resize(256) → CenterCrop(224) → ToTensor → Normalize`. Las augmentations se insertan **antes**
  de `ToTensor`/`Normalize` porque operan sobre imágenes PIL.
- **Transforms de validación (sin augmentation):** se mantiene el pipeline limpio
  `Resize(256) → CenterCrop(224) → ToTensor → Normalize`, porque en validación/evaluación
  **no** se aplican transformaciones aleatorias (queremos medir sobre imágenes deterministas).
- **Split reproducible:** usamos `torch.Generator().manual_seed(42)` con los **mismos índices**
  para train y val, tomando cada rama de un `ImageFolder` con su transform correspondiente. Así
  la única diferencia frente al entrenamiento base es la **augmentation en train**.
- **Modelo:** M2 con el mismo patrón de la Actividad 4 (VGG-16 preentrenado,
  `classifier = nn.Linear(25088, 2)`, `features` congeladas). Reutilizamos la función base `train`
  con `tag='aug'`.

> **Nota (local vs. Colab):** este experimento requiere los datos descargados y, en la práctica,
> **GPU**; está pensado para ejecutarse en Colab. En local, sin datos, esta celda no se ejecuta.

In [ ]:
# Experimento (pregunta 2): data augmentation. Celda AUTOCONTENIDA (no depende de ResNet).
# Construimos un pipeline de entrenamiento CON augmentations encima del pipeline base
# Resize(256)->CenterCrop(224)->ToTensor->Normalize, y dejamos la rama de validación SIN
# augmentation (pipeline limpio). Entrenamos un M2 (VGG-16 features congeladas + Linear) y
# registramos las curvas en TensorBoard con tag distinto 'aug' (runs/aug).

# 1) Transforms. 'std_normalize' ya está definido en la Sec. 2. Las augmentations operan sobre
#    imágenes PIL, por eso van ANTES de ToTensor()/Normalize.
trans_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    std_normalize,
])
# Validación: pipeline limpio (sin augmentation), igual que 'trans' de la Sec. 2.
trans_clean = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    std_normalize,
])

# 2) Dos vistas del mismo dataset con transforms distintos: una augmentada (train) y una
#    limpia (val). Aplicamos el transform por-split con el MISMO split reproducible (semilla 42),
#    para que la única diferencia frente al entrenamiento base sea la augmentation en train.
dataset_aug   = torchvision.datasets.ImageFolder(IMAGEFOLDER_ROOT, transform=trans_aug)
dataset_clean = torchvision.datasets.ImageFolder(IMAGEFOLDER_ROOT, transform=trans_clean)

n = len(dataset_aug)
n_train = 20000 if n > 20000 else int(0.8 * n)
# Mismos índices para ambas ramas (mismo generador con semilla 42).
idx_train, idx_test = torch.utils.data.random_split(
    range(n), [n_train, n - n_train],
    generator=torch.Generator().manual_seed(42)
)
trainset_aug  = torch.utils.data.Subset(dataset_aug,   list(idx_train))  # train CON augmentation
testset_clean = torch.utils.data.Subset(dataset_clean, list(idx_test))   # val SIN augmentation
print(f'Train (aug): {len(trainset_aug)} | Test (limpio): {len(testset_clean)}')

train_loader_aug = torch.utils.data.DataLoader(
    trainset_aug, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader_aug  = torch.utils.data.DataLoader(
    testset_clean, batch_size=16, num_workers=2, pin_memory=True)

# 3) Modelo M2 con el mismo patrón de la Actividad 4: VGG-16 preentrenado, classifier lineal
#    hacia 2 clases y features congeladas (solo se entrena la cabeza).
vgg_aug = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)
vgg_aug.classifier = nn.Linear(25088, 2)
for p in vgg_aug.features.parameters():
    p.requires_grad = False
vgg_aug = vgg_aug.to(device)
entrenables_aug = sum(p.numel() for p in vgg_aug.parameters() if p.requires_grad)
print(f'Parámetros entrenables (por código): {entrenables_aug}')

# 4) Entrenamiento reutilizando la función base `train`, con tag distinto 'aug' (runs/aug).
writer = SummaryWriter('runs/aug')
history_aug = train(
    vgg_aug, train_loader_aug, test_loader_aug,
    loss_fn=nn.CrossEntropyLoss(),
    epochs=1, lr=1e-3,
    writer=writer, tag='aug'
)
writer.close()

# 5) Curvas dibujadas dentro del notebook (reporte autocontenido).
plot_results(history_aug)

#### Conclusión del experimento de data augmentation (pregunta 2)

Compara las curvas del entrenamiento **con** data augmentation (`history_aug`, tag `aug`) frente
al entrenamiento **sin** augmentation (p. ej. el M2 congelado de la Sec. 6, tag `frozen`) en
TensorBoard (`%tensorboard --logdir runs`) y con las gráficas de `plot_results` dibujadas arriba.
Fíjate sobre todo en la **brecha entre train y validación**: si la augmentation ayuda a la
generalización, la curva de validación debería mantenerse más cerca de la de entrenamiento
(menos sobreajuste).

<!-- COMPLETAR tras ejecución -->
_(Respuesta empírica pendiente de ejecución en Colab. Tras ejecutar, describe aquí: si la data
augmentation redujo la brecha train/val (menos sobreajuste) o la aumentó, cómo cambió la accuracy
de validación respecto al entrenamiento sin augmentation, y si el impacto en la generalización
justifica el coste adicional por época.)_

## 8. Otras arquitecturas: ResNet

`torchvision.models` incluye muchas más redes preentrenadas. Una de las familias más usadas es **ResNet** (Microsoft Research), que resuelve el problema del gradiente desvaneciente en redes profundas mediante **conexiones residuales** (*skip connections*): cada bloque aprende una corrección a su entrada en lugar de una transformación desde cero.

Exploremos ResNet-18, la versión más ligera de la familia:


In [ ]:
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(resnet18)


Observa que la estructura es análoga a VGG: un extractor de features y una capa final de clasificación, aunque aquí se llama `fc` en lugar de `classifier`. También aparecen capas de **Batch Normalization** (`BatchNorm2d`), que normalizan las activaciones dentro de cada minibatch para estabilizar el entrenamiento.

> **TODO 6:** Adapta ResNet-18 para Cats vs. Dogs siguiendo el mismo esquema de la sección 6:
> 1. Congela todos los parámetros.
> 2. Reemplaza `resnet18.fc` por la capa adecuada (investiga cuántas entradas tiene).
> 3. Entrena 2 epochs y compara la velocidad y accuracy con VGG.


In [ ]:
# Carga de ResNet-18 con la API moderna de pesos (ResNet18_Weights.DEFAULT,
# sin el argumento deprecado pretrained=True) y lo movemos al device.
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)

# TODO 6a: Congela TODOS los parámetros de la red (backbone completo).
for p in resnet18.parameters():
    p.requires_grad = False

# TODO 6b: Reemplaza resnet18.fc. Determinamos POR CÓDIGO cuántas entradas tiene
# la capa final original (resnet18.fc.in_features) en vez de fijar el valor en
# texto, y la sustituimos por una Linear hacia 2 clases (Cats vs. Dogs). La nueva
# capa nace con requires_grad=True, así que solo ella se entrena.
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 2)
resnet18 = resnet18.to(device)
print(f'resnet18.fc.in_features (por código): {num_features}')
entrenables_rn = sum(p.numel() for p in resnet18.parameters() if p.requires_grad)
print(f'Parámetros entrenables de ResNet-18 (solo la cabeza fc): {entrenables_rn:,}')

# TODO 6c: Entrena 2 épocas con tag 'resnet18' en TensorBoard. Reutilizamos los
# splits trainset/testset de la Sec. 2; si los DataLoaders no existen aún en el
# entorno, los reconstruimos de forma defensiva (mismo patrón que las secciones
# anteriores).
if 'train_loader' not in globals() or 'test_loader' not in globals():
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

writer = SummaryWriter('runs/resnet18')
history_rn = train(
    resnet18, train_loader, test_loader,
    epochs=2, writer=writer, tag='resnet18'
)
writer.close()

# Curvas dibujadas con matplotlib dentro del notebook (reporte autocontenido).
plot_results(history_rn)


### Medición de tiempos por época: VGG-16 vs. ResNet-18

Para la **parte empírica de la Actividad 6** (¿ResNet-18 entrena más rápido
que VGG?) necesitamos cronometrar cuánto tarda **una época** de entrenamiento
en cada arquitectura, en igualdad de condiciones (mismo `device`, mismos
`DataLoader`, misma receta de transfer learning: backbone congelado y solo la
cabeza entrenable).

La función base `train` no expone los tiempos, así que **no modificamos su
firma**: en su lugar reutilizamos `train_epoch`/`eval_epoch` dentro de un bloque
cronometrado con `time.time()`, midiendo los segundos de **una época** por
modelo e imprimiéndolos para compararlos. Reconstruimos ambos modelos con la
misma estrategia de las secciones anteriores (VGG-16 con `classifier =
nn.Linear(25088, 2)` y `features` congeladas; ResNet-18 con `fc` reemplazado
determinando `in_features` por código y backbone congelado).

> Los tiempos reales dependen del hardware (GPU/CPU) y **solo se conocen al
> ejecutar en Colab**; por eso esta celda los **imprime** y la conclusión
> empírica se deja como plantilla a completar con esos números.


In [ ]:
# Medición del tiempo por época para comparar VGG-16 vs. ResNet-18 (Actividad 6).
# Cronometramos UNA época de entrenamiento por modelo con time.time(), sin
# alterar la firma de `train`: reutilizamos train_epoch/eval_epoch dentro de un
# bloque cronometrado. Los números reales se obtienen al ejecutar en Colab.
import time

# DataLoaders defensivos (mismo patrón que las secciones anteriores): si no
# existen en el entorno, los reconstruimos desde los splits de la Sec. 2.
if 'train_loader' not in globals() or 'test_loader' not in globals():
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

loss_fn = nn.CrossEntropyLoss()

# --- VGG-16 (M2 congelado): classifier -> Linear(25088, 2), features congeladas ---
vgg_timed = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)
vgg_timed.classifier = nn.Linear(25088, 2)
for p in vgg_timed.features.parameters():
    p.requires_grad = False
vgg_timed = vgg_timed.to(device)
opt_vgg = optim.Adam(filter(lambda p: p.requires_grad, vgg_timed.parameters()), lr=1e-3)

# --- ResNet-18 (M3): fc reemplazado (in_features por código), backbone congelado ---
resnet_timed = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in resnet_timed.parameters():
    p.requires_grad = False
resnet_timed.fc = nn.Linear(resnet_timed.fc.in_features, 2)
resnet_timed = resnet_timed.to(device)
opt_rn = optim.Adam(filter(lambda p: p.requires_grad, resnet_timed.parameters()), lr=1e-3)

# Cronometramos UNA época de cada modelo con time.time() alrededor de la llamada.
t0 = time.time()
train_epoch(vgg_timed, train_loader, opt_vgg, loss_fn)
eval_epoch(vgg_timed, test_loader, loss_fn)
tiempo_vgg = time.time() - t0

t0 = time.time()
train_epoch(resnet_timed, train_loader, opt_rn, loss_fn)
eval_epoch(resnet_timed, test_loader, loss_fn)
tiempo_resnet = time.time() - t0

# Impresión de los tiempos medidos para la comparación.
print(f'Tiempo por época VGG-16     : {tiempo_vgg:.2f} s')
print(f'Tiempo por época ResNet-18  : {tiempo_resnet:.2f} s')
if tiempo_resnet > 0:
    print(f'VGG-16 / ResNet-18 (ratio) : {tiempo_vgg / tiempo_resnet:.2f}x')
mas_rapido = 'ResNet-18' if tiempo_resnet < tiempo_vgg else 'VGG-16'
print(f'Modelo más rápido por época : {mas_rapido}')


#### Respuesta empírica (Actividad 6): ¿ResNet-18 entrena más rápido que VGG y por qué?

<!-- COMPLETAR tras ejecución -->

Completa esta respuesta con los **tiempos por época** que imprime la celda
anterior (`tiempo_vgg` y `tiempo_resnet`):

- Tiempo por época de **VGG-16**: _____ s (ver salida de la celda).
- Tiempo por época de **ResNet-18**: _____ s (ver salida de la celda).
- Modelo más rápido por época: _____ (según los tiempos medidos).

Interpretación esperada (confirmar con los números reales): ResNet-18 tiene
**muchos menos parámetros** que VGG-16 (sobre todo porque VGG concentra la mayor
parte de sus pesos en el clasificador totalmente conectado) y usa **global
average pooling** en lugar de capas densas enormes, por lo que suele requerir
menos cómputo y memoria por paso. Indica si los tiempos medidos confirman que
ResNet-18 entrena más rápido por época y razona la causa a partir de la salida.


### Pregunta (Actividad 6): ¿qué es un *residual block* y qué problema resuelve en redes profundas?

**Qué es un residual block (bloque residual).** Es la unidad básica de las redes **ResNet**. En lugar de pedirle a un bloque de capas que aprenda directamente la transformación deseada `H(x)` a partir de su entrada `x`, el bloque aprende solo la **diferencia** (el *residuo*) `F(x) = H(x) - x`, y su salida se reconstruye sumando esa corrección a la entrada: `salida = F(x) + x`. Esa suma se implementa con una **conexión de atajo** (*skip connection* o *shortcut*) que lleva la entrada `x` directamente hasta la salida del bloque, saltándose las capas intermedias (típicamente dos convoluciones con BatchNorm y ReLU). Cuando las dimensiones de `x` y de `F(x)` no coinciden (por un cambio de número de canales o de resolución), el atajo aplica una proyección lineal (una convolución 1×1) para igualarlas antes de sumar.

**Qué problema resuelve.** Al apilar muchas capas, las redes profundas "planas" (como VGG) sufren dos dificultades relacionadas:

1. **Gradientes que se desvanecen (*vanishing gradients*).** Durante la retropropagación, el gradiente se multiplica capa a capa; en redes muy profundas puede encogerse hasta casi anularse al llegar a las primeras capas, que entonces casi no aprenden. La conexión de atajo crea un **camino directo** para el gradiente (la derivada de `F(x) + x` respecto a `x` incluye un término `1` que "deja pasar" el gradiente sin atenuarlo), de modo que el entrenamiento de redes muy profundas se vuelve estable.
2. **Degradación (*degradation problem*).** Empíricamente se observó que, a partir de cierta profundidad, añadir más capas a una red plana **empeora** el error de entrenamiento (no por overfitting, sino por la dificultad de optimización). Con bloques residuales, si la transformación óptima para un bloque es la **identidad**, a la red le basta con llevar `F(x)` hacia 0, algo mucho más fácil de aprender que reproducir la identidad con un apilado de capas no lineales. Así, añadir profundidad **no degrada** el rendimiento y, en la práctica, permite entrenar redes de decenas o cientos de capas que superan a las arquitecturas planas equivalentes.

> En resumen: un residual block reformula el aprendizaje como "aprender una corrección sobre la entrada" mediante *skip connections*. Esto facilita el flujo del gradiente (mitiga el *vanishing gradient*) y resuelve el problema de **degradación** al hacer trivial representar la identidad, habilitando el entrenamiento efectivo de redes muy profundas.


## 9. Comparación de experimentos con TensorBoard

Todos los experimentos anteriores escribieron logs en el directorio `runs/`. TensorBoard permite comparar curvas de loss y accuracy de todos los experimentos en un mismo panel.

**Cada experimento usa un tag/subdirectorio DISTINTO bajo `runs/`**, de modo que sus curvas aparecen como series separadas y comparables en TensorBoard:

| Experimento | Sección | Directorio del `SummaryWriter` | Tag |
|---|---|---|---|
| Clasificador lineal sobre features | Sec. 5 | `runs/linear_classifier` | `linear` |
| VGG-16 con `features` congeladas | Sec. 6 | `runs/frozen` | `frozen` |
| VGG-16 fine-tuned (lr=1e-4) | Sec. 7 | `runs/finetuned` | `finetuned` |
| VGG-16 fine-tuned + scheduler | Sec. 7 | `runs/finetuned_sched` | `finetuned_sched` |
| Data augmentation (M2) | Sec. 7 | `runs/aug` | `aug` |
| ResNet-18 adaptado | Sec. 8 | `runs/resnet18` | `resnet18` |

Como los seis directorios son distintos, no hay colisión de curvas: TensorBoard las lista por separado y pueden superponerse para compararlas.

**Reporte autocontenido:** además de TensorBoard, cada experimento dibuja sus curvas de loss/accuracy con `plot_results(...)` (matplotlib) dentro del propio notebook, justo tras su entrenamiento. Así el reporte queda completo aunque el panel de TensorBoard no se conserve al compartir el `.ipynb` ejecutado.

La celda siguiente carga la extensión de TensorBoard y abre el panel apuntando a `runs/`:


In [ ]:
# En terminal:
#   tensorboard --logdir=runs --port=6006

# En Jupyter / Colab:
%load_ext tensorboard
%tensorboard --logdir runs


## (OPCIONAL) Reto EfficientNet-B0 (pregunta de investigación 8)

> **Esta sección es OPCIONAL.** Corresponde al **reto EfficientNet-B0** (pregunta de investigación **8** del enunciado). No forma parte de la entrega obligatoria y puede omitirse sin afectar las Actividades 1 a 6.

**EfficientNet-B0** es una arquitectura más moderna que VGG-16 y ResNet-18. Su diseño escala de forma equilibrada la profundidad, el ancho y la resolución de la red (*compound scaling*), de modo que alcanza una accuracy competitiva en ImageNet con **muchos menos parámetros y menos cómputo**. Aquí la usamos como un tercer backbone de *transfer learning* para el problema binario Cats vs. Dogs, siguiendo exactamente el mismo esquema que ResNet-18 (Sec. 8):

- **Carga con la API moderna de pesos.** Usamos `torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT)` (nunca el argumento deprecado `pretrained=True`), para obtener los pesos preentrenados en ImageNet.
- **Congelar el backbone.** Ponemos `requires_grad = False` en todos los parámetros del extractor de características (`model.features`), de modo que esas capas convolucionales actúen como extractor fijo y **no** se entrenen.
- **Adaptar el clasificador final determinando sus entradas POR CÓDIGO.** En EfficientNet, la cabeza es un `nn.Sequential` llamado `model.classifier` cuyo **último** módulo es la capa lineal de salida. En lugar de fijar en texto cuántas entradas tiene, las leemos por código con `in_features = model.classifier[-1].in_features` y reemplazamos esa última capa por `nn.Linear(in_features, 2)` (dos clases: gato/perro). La nueva capa nace con `requires_grad=True`, así que **solo ella** se entrena.
- **Mover al `device` y contar entrenables por código.** Llevamos el modelo a `device` (GPU en Colab) e imprimimos el número de parámetros entrenables con `sum(p.numel() for p in model.parameters() if p.requires_grad)`, sin afirmar ningún valor en texto.
- **Entrenamiento breve y logging comparable.** Reutilizamos la función base `train` por unas pocas épocas, registrando las curvas en TensorBoard con un **tag distinto** (`efficientnet`, bajo `runs/efficientnet`) para que no colisionen con los demás experimentos, y dibujamos las curvas con `plot_results` para que el reporte sea autocontenido.

> **Nota (local vs. Colab):** esta sección descarga pesos preentrenados y entrena sobre los datos, por lo que requiere los datos descargados y, en la práctica, **GPU**; está pensada para ejecutarse en **Colab**. En local, sin datos ni pesos, esta celda no se ejecuta. La **comparación** de accuracy final y tiempo de entrenamiento de EfficientNet-B0 frente a ResNet-18 y VGG-16 se aborda en la celda siguiente.


In [ ]:
# (OPCIONAL) EfficientNet-B0 como backbone de transfer learning (pregunta 8).
# Mismo esquema que ResNet-18 (Sec. 8): cargar con pesos modernos, congelar el
# backbone, adaptar la cabeza determinando sus entradas POR CÓDIGO, mover al
# device, contar entrenables por código y entrenar brevemente con tag propio.

# 1) Carga con la API moderna de pesos (sin el argumento deprecado pretrained=True).
effnet = torchvision.models.efficientnet_b0(
    weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
)

# 2) Congelar el backbone (extractor de características): no se entrena.
for p in effnet.features.parameters():
    p.requires_grad = False

# 3) Adaptar el clasificador final determinando sus entradas POR CÓDIGO.
#    En EfficientNet la cabeza es un nn.Sequential; su ÚLTIMO módulo es la capa
#    lineal de salida. Leemos sus entradas con classifier[-1].in_features (no en
#    texto) y la reemplazamos por una Linear hacia 2 clases (Cats vs. Dogs).
in_features = effnet.classifier[-1].in_features
effnet.classifier[-1] = nn.Linear(in_features, 2)

# 4) Mover al device (GPU en Colab).
effnet = effnet.to(device)

# 5) Conteo de parámetros entrenables POR CÓDIGO (solo la nueva cabeza), impreso
#    en lugar de afirmarse en texto.
print(f'effnet.classifier[-1].in_features (por código): {in_features}')
entrenables_eff = sum(p.numel() for p in effnet.parameters() if p.requires_grad)
print(f'Parámetros entrenables de EfficientNet-B0 (solo la cabeza): {entrenables_eff:,}')

# 6) Entrenamiento breve con tag distinto 'efficientnet' en TensorBoard.
#    DataLoaders defensivos (mismo patrón que las secciones anteriores).
if 'train_loader' not in globals() or 'test_loader' not in globals():
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=16, num_workers=2, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(testset,  batch_size=16, num_workers=2, pin_memory=True)

writer = SummaryWriter('runs/efficientnet')
history_eff = train(
    effnet, train_loader, test_loader,
    epochs=2, writer=writer, tag='efficientnet'
)
writer.close()

# Curvas dibujadas con matplotlib dentro del notebook (reporte autocontenido).
plot_results(history_eff)


### (OPCIONAL) Comparación de accuracy y tiempo: EfficientNet-B0 vs. ResNet-18 vs. VGG-16

> **Esta subsección sigue siendo OPCIONAL** (reto EfficientNet-B0, pregunta de investigación **8**). Cierra la comparación pedida por el enunciado: **accuracy final** y **tiempo de entrenamiento** de EfficientNet-B0 frente a ResNet-18 y VGG-16.

La celda de código siguiente:

- **Mide el tiempo por época de EfficientNet-B0** con `time.time()`, cronometrando **una época** de entrenamiento/validación (reutilizando `train_epoch`/`eval_epoch`), igual que la celda de tiempos de la Sec. 8 hizo para VGG-16 y ResNet-18. El resultado se guarda en `tiempo_eff`.
- **Toma la accuracy final de validación de cada modelo** del último valor de su historial de entrenamiento: `history_eff` (EfficientNet-B0), `history_rn` (ResNet-18) y el fine-tuning de VGG-16 (`history_ft`, o `history_frozen` si el fine-tuning no se ejecutó). No se afirma ningún número en texto: se **leen por código** de los historiales ya calculados.
- **Reúne los tiempos por época** ya medidos en la Sec. 8 (`tiempo_vgg`, `tiempo_resnet`) junto con `tiempo_eff`.
- **Imprime una tabla comparativa** (accuracy final + tiempo por época) de los tres modelos. La construcción es **defensiva**: si alguna sección previa no se ejecutó (p. ej. en local, sin datos ni pesos), la variable correspondiente no existirá y la tabla mostrará `N/D` (no disponible) en lugar de inventar un valor.

> **Anti-invención:** todos los valores empíricos (accuracies y tiempos) se **miden/imprimen por código** al ejecutar en Colab. La conclusión final queda como **plantilla** con el marcador `<!-- COMPLETAR tras ejecución -->` en la celda posterior, para rellenarla con los números reales que imprima esta celda.

> **Nota (local vs. Colab):** medir `tiempo_eff` requiere datos y pesos descargados (y, en la práctica, **GPU**), por lo que esta celda está pensada para **Colab**. En local, sin datos, el bloque de medición se omite de forma segura e informa de ello.


In [ ]:
# (OPCIONAL) Comparación de accuracy final y tiempo de entrenamiento:
# EfficientNet-B0 vs. ResNet-18 vs. VGG-16 (pregunta de investigación 8).
# Mide el tiempo por época de EfficientNet-B0 con time.time() y arma una tabla
# con la accuracy final (leída de los historiales) y los tiempos por época.
# NO inventa valores: lo que no se haya calculado se muestra como 'N/D'.
import time


def _acc_final(history):
    """Devuelve la última accuracy de validación de un history, o None."""
    if isinstance(history, dict) and history.get('val_acc'):
        return history['val_acc'][-1]
    return None


# 1) Medir el tiempo por época de EfficientNet-B0 con time.time() (una época),
#    reutilizando train_epoch/eval_epoch, igual que la celda de tiempos de la
#    Sec. 8. Requiere datos + pesos (Colab); si no, se omite de forma segura.
tiempo_eff = None
if ('effnet' in globals() and 'train_loader' in globals()
        and 'test_loader' in globals()):
    _loss_fn = nn.CrossEntropyLoss()
    _opt_eff = optim.Adam(
        filter(lambda p: p.requires_grad, effnet.parameters()), lr=1e-3
    )
    _t0 = time.time()
    train_epoch(effnet, train_loader, _opt_eff, _loss_fn)
    eval_epoch(effnet, test_loader, _loss_fn)
    tiempo_eff = time.time() - _t0
    print(f'Tiempo por época EfficientNet-B0: {tiempo_eff:.2f} s')
else:
    print('[AVISO] No se midió tiempo_eff: faltan effnet/train_loader/test_loader.')
    print('        (Esperado en local: sin datos ni pesos. Mídelo en Colab.)')

# 2) Accuracy final de validación de cada modelo (leída por código de los
#    historiales ya calculados; VGG-16 usa el fine-tuning, con fallback al
#    congelado si el fine-tuning no se ejecutó). None si no está disponible.
acc_eff = _acc_final(globals().get('history_eff'))
acc_rn  = _acc_final(globals().get('history_rn'))
acc_vgg = _acc_final(globals().get('history_ft'))
if acc_vgg is None:
    acc_vgg = _acc_final(globals().get('history_frozen'))

# 3) Tiempos por época: EfficientNet (medido arriba) + los de la Sec. 8.
tiempo_vgg_val    = globals().get('tiempo_vgg')
tiempo_resnet_val = globals().get('tiempo_resnet')

# 4) Tabla comparativa. 'N/D' = no disponible (sección no ejecutada): NO se
#    inventa ningún número; se completa al ejecutar en Colab.
def _fmt_acc(a):
    return f'{a:.4f}' if isinstance(a, (int, float)) else 'N/D'


def _fmt_t(t):
    return f'{t:.2f} s' if isinstance(t, (int, float)) else 'N/D'


filas = [
    ('EfficientNet-B0', acc_eff, tiempo_eff),
    ('ResNet-18',       acc_rn,  tiempo_resnet_val),
    ('VGG-16',          acc_vgg, tiempo_vgg_val),
]

print()
print(f"{'Modelo':<16} | {'Acc. final (val)':<16} | {'Tiempo/época':<12}")
print('-' * 52)
for nombre, acc, t in filas:
    print(f'{nombre:<16} | {_fmt_acc(acc):<16} | {_fmt_t(t):<12}')

# Marcador para localizar esta comparación (comparar_accuracy_tiempo).
print('\n[comparar_accuracy_tiempo] Completa la conclusión con estos valores.')


#### Conclusión de la comparación (OPCIONAL) — pregunta de investigación 8

<!-- COMPLETAR tras ejecución -->

Completa esta conclusión con los **valores reales** que imprime la celda anterior (accuracy final de validación y tiempo por época de cada modelo):

| Modelo | Accuracy final (val) | Tiempo por época |
|---|---|---|
| **EfficientNet-B0** | _____ (ver `history_eff` / salida) | _____ s (ver `tiempo_eff`) |
| **ResNet-18** | _____ (ver `history_rn` / salida) | _____ s (ver `tiempo_resnet`) |
| **VGG-16** | _____ (ver `history_ft`/`history_frozen` / salida) | _____ s (ver `tiempo_vgg`) |

A partir de esos números, responde de forma concreta:

- **Accuracy:** ¿qué modelo alcanzó la mejor accuracy final de validación en este problema binario (Cats vs. Dogs)? ¿Las diferencias son grandes o marginales?
- **Tiempo:** ¿qué modelo entrena más rápido por época? Ordénalos de más rápido a más lento según los tiempos medidos.
- **Compromiso accuracy/tiempo:** EfficientNet-B0 está diseñado para lograr buena accuracy con **menos parámetros y menos cómputo** (*compound scaling*). Indica si los números medidos confirman esa relación coste/beneficio frente a ResNet-18 y VGG-16, y cuál elegirías para este caso y por qué.

> **Nota:** no inventes valores. Rellena la tabla únicamente con los números que imprima la celda de código anterior al ejecutar el notebook en Colab.


## 10. Preguntas de investigación

Esta sección responde las **8 preguntas de investigación** del enunciado **en su numeración original** (la 8, EfficientNet-B0, es un reto **opcional**).

Clasificamos cada pregunta según su tipo:

- **Teórica:** la respuesta no depende de ejecutar el notebook; se responde de forma completa aquí en Markdown.
- **Empírica:** la respuesta depende de resultados de ejecución (accuracy, tiempos, comparación de curvas). Se referencia el experimento correspondiente y la conclusión numérica se deja como **plantilla** con el marcador `<!-- COMPLETAR tras ejecución -->`, a rellenar tras el *Run all* en Colab.

> **Nota (preguntas 3 y 7).** Las preguntas 3 y 7 del `.md` son casi idénticas (ambas sobre *learning rate schedulers* `StepLR`/`CosineAnnealingLR` en el fine-tuning). La **respuesta sustantiva se da una sola vez en la pregunta 3**; la pregunta 7 la **referencia explícitamente** sin duplicar contenido.


### Pregunta 1 (teórica): ¿cuándo NO conviene usar transfer learning desde ImageNet?

El *transfer learning* desde ImageNet funciona porque los filtros aprendidos (bordes, texturas, formas) son útiles para **muchas** tareas de visión sobre **fotografías naturales**. Pero esa misma razón explica cuándo **no** es beneficioso: cuando el dominio de destino se parece poco a ImageNet, o cuando las condiciones del problema anulan la ventaja del preentrenamiento. A continuación, al menos dos escenarios concretos.

**Escenario 1 — Dominio muy distinto al de ImageNet (desajuste de dominio).** Si las imágenes de destino no son fotografías naturales RGB, los filtros preentrenados aportan poco o pueden incluso estorbar. Ejemplos concretos:

- **Imágenes médicas** como radiografías, resonancias o cortes histopatológicos: son en escala de grises o con texturas/estadísticas muy diferentes a las fotos de ImageNet; los bordes y colores aprendidos en ImageNet no capturan bien las estructuras clínicas relevantes.
- **Imágenes satelitales/multiespectrales** (más de 3 canales, p. ej. infrarrojo) o **datos no fotográficos** representados como imagen (espectrogramas de audio, mapas de sensores): la entrada ni siquiera coincide con los 3 canales RGB que espera el backbone, y las correlaciones espaciales son distintas.

En estos casos, un modelo preentrenado en ImageNet suele requerir un fine-tuning agresivo (que diluye la ventaja) o directamente rinde peor que entrenar una arquitectura adaptada al dominio.

**Escenario 2 — Dataset de destino muy grande y representativo (la ventaja se desvanece).** Cuando dispones de **muchísimos** datos etiquetados y bien distribuidos para tu tarea, entrenar desde cero puede igualar o superar al transfer learning, porque el modelo aprende filtros **específicos** de tu dominio sin el sesgo de los pesos de ImageNet. El preentrenamiento ayuda sobre todo cuando los datos son **escasos**; con datos abundantes su valor marginal cae.

**Escenario 3 (adicional) — Restricciones de arquitectura o despliegue.** Si necesitas una arquitectura muy pequeña o personalizada (p. ej. por límites de memoria/latencia en un dispositivo embebido) que no tiene pesos de ImageNet disponibles, forzar un backbone grande y preentrenado solo para "reutilizar pesos" puede ser contraproducente frente a entrenar un modelo ligero a medida.

> **Síntesis:** transfer learning desde ImageNet **no** conviene cuando (a) el dominio de destino difiere mucho del de ImageNet (imágenes médicas, satelitales/multiespectrales, datos no fotográficos) o (b) se dispone de un dataset de destino suficientemente grande y representativo como para que entrenar desde cero sea competitivo; y es cuestionable cuando (c) restricciones de arquitectura/despliegue impiden aprovechar un backbone preentrenado.


### Pregunta 2 (empírica): data augmentation y su impacto en la generalización

**Pregunta del `.md`:** ¿qué transforms adicionales añadirías al pipeline de entrenamiento para mejorar la generalización? Impleméntalos y mide el impacto en TensorBoard.

**Experimento implementado (ver sección de *data augmentation*, tag `aug`).** El experimento define un pipeline de **entrenamiento** con augmentations —`RandomHorizontalFlip`, `RandomRotation` y `ColorJitter`— manteniendo el pipeline de **validación** sin aumentar (`Resize(256) → CenterCrop(224) → ToTensor → Normalize`), de modo que la comparación sea justa. Las curvas se registran en TensorBoard bajo `runs/aug` (tag distinto) y se dibujan con `plot_results` dentro del notebook.

**Qué se espera observar (marco de análisis).** La data augmentation aumenta artificialmente la variedad del conjunto de entrenamiento (reflejos, pequeñas rotaciones, cambios de brillo/contraste/saturación), lo que suele **reducir el sobreajuste**: típicamente se observa una brecha menor entre *accuracy* de entrenamiento y de validación, y una *accuracy* de validación igual o mejor que sin augmentations (a costa, a veces, de una *accuracy* de entrenamiento algo menor y de necesitar más épocas para converger).

**Conclusión empírica (a completar con los números del experimento):**

<!-- COMPLETAR tras ejecución -->

- *accuracy* de validación **sin** augmentation (baseline, tag `frozen`/`finetuned`): \_\_\_
- *accuracy* de validación **con** augmentation (tag `aug`): \_\_\_
- Brecha train–val observada con vs. sin augmentation y conclusión sobre generalización: \_\_\_

> Rellena estos valores leyendo las curvas de TensorBoard (`runs/aug` vs. el baseline) y las gráficas de `plot_results` tras el *Run all* en Colab. No inventes valores.


### Pregunta 3 (empírica): learning rate scheduler en el fine-tuning (respuesta sustantiva única)

**Pregunta del `.md`:** investiga `torch.optim.lr_scheduler.StepLR` y `CosineAnnealingLR`; agrega uno al loop de fine-tuning y compara las curvas.

**Qué es un *learning rate scheduler* y por qué ayuda en el fine-tuning.** Un *scheduler* ajusta el *learning rate* (lr) a lo largo del entrenamiento en lugar de mantenerlo fijo. En *fine-tuning* esto es especialmente útil: se parte de pesos ya buenos (preentrenados + cabeza estabilizada), así que conviene **reducir el lr gradualmente** para hacer ajustes cada vez más finos sin desestabilizar el backbone.

- **`StepLR(optimizer, step_size, gamma)`** reduce el lr de forma **escalonada**: cada `step_size` épocas lo multiplica por `gamma` (p. ej. ×0.1). Produce caídas bruscas en instantes concretos; es simple y predecible.
- **`CosineAnnealingLR(optimizer, T_max)`** hace decaer el lr siguiendo **medio ciclo de coseno** desde el valor inicial hasta (casi) 0 a lo largo de `T_max` épocas. El descenso es **suave y continuo**, lo que suele ayudar a converger a mejores mínimos y es muy habitual en fine-tuning.

**Experimento implementado (ver sección del *scheduler*, tag `finetuned_sched`).** Como la función base `train` no expone un parámetro de *scheduler*, el experimento usa un **loop de fine-tuning dedicado** que reutiliza `train_epoch`/`eval_epoch` y añade el *scheduler* (`scheduler.step()` por época), registrando las curvas en TensorBoard bajo `runs/finetuned_sched` (tag distinto) y dibujándolas con `plot_results`. Así se pueden **comparar** las curvas del fine-tuning **con** scheduler frente al fine-tuning base **sin** scheduler (tag `finetuned`).

**Conclusión empírica (a completar con los números del experimento):**

<!-- COMPLETAR tras ejecución -->

- *scheduler* elegido (`StepLR` o `CosineAnnealingLR`) y sus hiperparámetros: \_\_\_
- *accuracy*/loss de validación **sin** scheduler (tag `finetuned`) vs. **con** scheduler (tag `finetuned_sched`): \_\_\_
- ¿El scheduler mejoró la convergencia o la *accuracy* final? Describe la diferencia observada en las curvas: \_\_\_

> Rellena estos valores comparando `runs/finetuned` vs. `runs/finetuned_sched` en TensorBoard y las gráficas de `plot_results` tras el *Run all* en Colab. No inventes valores.


### Pregunta 4 (teórica): ¿por qué ResNet usa BatchNorm y VGG no, y qué efecto tiene en el entrenamiento?

**Contexto histórico.** VGG-16 (2014) es anterior a la popularización de la *Batch Normalization*, que se introdujo en 2015. Por eso la arquitectura VGG original **no** incluye capas de BatchNorm (apila convoluciones + ReLU + max-pooling). ResNet (2015) se diseñó **ya con** BatchNorm como componente estándar: tras cada convolución del bloque residual aplica `BatchNorm` antes de la activación. (Existe una variante `vgg16_bn` en torchvision que añade BatchNorm a VGG, pero la VGG "clásica" no la tiene.)

**Qué hace BatchNorm.** Normaliza las activaciones de cada capa por *mini-batch* (resta la media y divide por la desviación estándar del batch) y luego las reescala con parámetros aprendibles `gamma`/`beta`. Esto estabiliza la **distribución** de las activaciones a lo largo del entrenamiento.

**Efecto en el entrenamiento (por qué importa, sobre todo en redes profundas como ResNet):**

- **Permite entrenar redes mucho más profundas.** ResNet se diseñó para decenas o cientos de capas; sin normalización, las activaciones y gradientes tienden a crecer o desvanecerse al propagarse por tantas capas. BatchNorm mantiene las activaciones en un rango estable, lo que —junto con las conexiones residuales— hace entrenable esa profundidad.
- **Acelera la convergencia y permite *learning rates* mayores.** Al estabilizar las distribuciones internas, el entrenamiento tolera pasos más grandes y suele converger en menos épocas.
- **Actúa como regularizador suave.** El ruido introducido por las estadísticas por *batch* tiene un leve efecto regularizador, reduciendo algo la dependencia de otras técnicas.

**Consecuencia práctica para VGG.** Al no tener BatchNorm, VGG es **más sensible a la inicialización y al learning rate**, y su gran clasificador denso depende más de Dropout para regularizar. En redes muy profundas, la ausencia de normalización haría el entrenamiento inestable; de ahí que las arquitecturas modernas (ResNet y posteriores) la incorporen de serie.


### Pregunta 5 (teórica): origen de la media/std de ImageNet y qué pasaría con otros valores

**Origen de los valores.** Los vectores `mean = [0.485, 0.456, 0.406]` y `std = [0.229, 0.224, 0.225]` (en orden de canales RGB) son las **estadísticas calculadas sobre el conjunto de entrenamiento de ImageNet**: se tomaron todas las imágenes de entrenamiento, se escalaron sus píxeles al rango `[0, 1]` (como hace `ToTensor()`) y se calcularon, **por canal**, la media y la desviación estándar de los valores de los píxeles. No son números arbitrarios: resumen "cómo luce en promedio" un píxel de ImageNet en cada canal. Esta respuesta amplía la ya dada en la Actividad 1 (Sec. 2).

**Qué hace la normalización.** `transforms.Normalize(mean, std)` aplica, canal a canal, `x_norm = (x - mean) / std`, dejando cada canal **centrado en 0** y con **escala comparable** (varianza cercana a 1). Es exactamente la transformación que recibieron las imágenes cuando el modelo (VGG-16, ResNet-18, ...) se entrenó originalmente en ImageNet.

**Qué pasaría al usar otros valores:**

- **No normalizar (o normalizar con media/std muy distintas):** las activaciones internas quedan **fuera del rango esperado** por los pesos preentrenados. Los filtros, calibrados para una entrada centrada y escalada de cierta forma, responden de manera subóptima; las representaciones intermedias se degradan y tanto la inferencia directa como el *transfer learning* (extracción de features o fine-tuning) parten de peores representaciones. En la práctica, la *accuracy* cae y la convergencia empeora.
- **Usar la media/std del propio dataset de destino:** es una opción **válida** cuando se **entrena desde cero**, porque entonces la red aprende sus pesos acorde a esa normalización. Pero al reutilizar pesos preentrenados de ImageNet lo coherente es usar la media/std **de ImageNet**, para que la distribución de entrada coincida con la que la red ya sabe procesar.

> **Síntesis:** la media/std provienen de las estadísticas del set de entrenamiento de ImageNet; usarlas es imprescindible al emplear pesos preentrenados porque esos pesos se ajustaron sobre entradas normalizadas exactamente así. Cambiarlas desalinea la distribución de entrada respecto a la que la red espera y deteriora el rendimiento.


### Pregunta 6 (teórica): diferencia entre `CrossEntropyLoss`, `NLLLoss` y `LogSoftmax`, y por qué no añadir `LogSoftmax`

**`nn.LogSoftmax`** es una **capa/activación**, no una pérdida. Aplica *softmax* a los *logits* y luego el logaritmo: convierte los *logits* crudos en **log-probabilidades** (valores ≤ 0 que, exponenciados, suman 1). No calcula ningún error respecto a la etiqueta.

**`nn.NLLLoss`** (Negative Log-Likelihood Loss) es una **función de pérdida** que espera recibir **log-probabilidades** (la salida de `LogSoftmax`) y las etiquetas enteras. Simplemente toma el valor de la log-probabilidad de la clase correcta y lo niega. Por sí sola **no** aplica softmax ni logaritmo: asume que ya se los dieron hechos.

**`nn.CrossEntropyLoss`** es una **función de pérdida "todo en uno"** que combina internamente `LogSoftmax` **+** `NLLLoss`. Es decir, recibe directamente los **logits crudos** (la salida lineal de la red, sin activación final) y las etiquetas, y hace internamente el `log_softmax` seguido del *negative log-likelihood*. Matemáticamente: `CrossEntropyLoss(logits, y) == NLLLoss(LogSoftmax(logits), y)`.

**Por qué NO se debe añadir `LogSoftmax` a la red si ya se usa `CrossEntropyLoss`.** Porque `CrossEntropyLoss` **ya aplica `LogSoftmax` internamente**. Si la última capa de la red fuese un `LogSoftmax` (o un `Softmax`) y además se usa `CrossEntropyLoss`, se estaría aplicando la normalización **dos veces**:

- La transformación queda mal definida (log-softmax de un log-softmax), los **gradientes** resultantes son incorrectos y el entrenamiento se vuelve inestable o aprende mal.
- Se pierde además la **estabilidad numérica** que `CrossEntropyLoss` logra al fusionar `log` y `softmax` en una sola operación (el *log-sum-exp trick*).

**Regla práctica (coherente con M1 en la Actividad 3):** la red debe terminar en una **capa lineal que produzca logits crudos** (`nn.Linear(..., num_clases)` **sin** activación final) y usar `nn.CrossEntropyLoss`. La alternativa equivalente sería terminar en `LogSoftmax` y usar `NLLLoss`, pero **nunca** combinar `LogSoftmax` con `CrossEntropyLoss`.


### Pregunta 7 (empírica): learning rate scheduling (referencia a la pregunta 3)

La pregunta 7 del `.md` (investigar `StepLR`/`CosineAnnealingLR`, agregar uno al loop de fine-tuning y comparar curvas) es **esencialmente la misma** que la pregunta 3. Por tanto, **no se duplica** la respuesta:

- La **explicación teórica** de `StepLR` vs. `CosineAnnealingLR` y de por qué un scheduler ayuda en el fine-tuning está en la **Pregunta 3** de esta sección.
- El **experimento** (loop de fine-tuning dedicado con scheduler, tag `finetuned_sched`, comparación de curvas en TensorBoard y `plot_results`) también se describe y se implementa en la **Pregunta 3** / la sección del scheduler.
- La **conclusión empírica** correspondiente es la misma plantilla de la Pregunta 3 (ver `<!-- COMPLETAR tras ejecución -->` allí); se completa una sola vez con los números de `runs/finetuned` vs. `runs/finetuned_sched` tras el *Run all* en Colab.


### Pregunta 8 (OPCIONAL, empírica): EfficientNet-B0 vs. ResNet-18 vs. VGG-16

**Pregunta del `.md` (reto opcional):** usa `EfficientNet-B0` (disponible en `torchvision.models`) y compara *accuracy* final y tiempo de entrenamiento contra ResNet-18 y VGG-16.

**Sección implementada (ver la sección OPCIONAL de EfficientNet-B0).** Allí se carga `efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)` con la API moderna, se congela el backbone y se adapta el clasificador final determinando sus entradas **por código** (`model.classifier[-1].in_features`). La comparación de *accuracy* y tiempo frente a ResNet-18 y VGG-16 se mide con el código de esa sección.

**Marco de la comparación.** EfficientNet-B0 aplica *compound scaling* (equilibra profundidad, anchura y resolución) para lograr buena *accuracy* con **menos parámetros y menos cómputo** que redes más antiguas; la pregunta empírica es si, en este problema binario (Cats vs. Dogs), ese compromiso coste/beneficio se confirma frente a ResNet-18 y VGG-16.

**Conclusión empírica (a completar con los números del experimento):**

<!-- COMPLETAR tras ejecución -->

| Modelo | Accuracy final (val) | Tiempo por época | Nº parámetros entrenables |
|---|---|---|---|
| VGG-16 (M2) | \_\_\_ | \_\_\_ | \_\_\_ |
| ResNet-18 (M3) | \_\_\_ | \_\_\_ | \_\_\_ |
| EfficientNet-B0 | \_\_\_ | \_\_\_ | \_\_\_ |

- ¿Confirma EfficientNet-B0 un mejor compromiso accuracy/coste? ¿Cuál elegirías para este caso y por qué?: \_\_\_

> Rellena la tabla únicamente con los números que impriman las celdas de código al ejecutar en Colab. No inventes valores. (Pregunta **opcional**: puede omitirse sin afectar la entrega obligatoria.)


## Apéndice: verificación de arquitecturas

Esta celda es una **verificación de arquitecturas** pensada para ejecutarse **en local**, donde no hay
GPU, ni el dataset de Kaggle, ni los pesos preentrenados descargados. Su único
objetivo es comprobar que la **arquitectura** de los modelos de transfer learning
queda bien armada antes de llevar el notebook a Colab.

Para lograrlo:

- Usamos `weights=None`, de modo que **no se descarga ningún peso preentrenado**.
- Alimentamos los modelos con **tensores aleatorios** de forma `[B, 3, 224, 224]`,
  así que **no se necesita el dataset** ni ninguna imagen real.

Verificamos por código, sin afirmar valores en texto:

- **M2 (VGG-16):** reemplazamos `classifier` por `nn.Linear(25088, 2)` y congelamos
  `features`. Comprobamos que la salida tiene forma `(B, 2)` e imprimimos el número
  de parámetros entrenables (`requires_grad == True`).
- **M3 (ResNet-18):** reemplazamos `fc` determinando `in_features` **por código** y
  congelamos el backbone (solo `fc` queda entrenable). Comprobamos que la salida
  tiene forma `(B, 2)` e imprimimos el número de parámetros entrenables.

Si ambos `assert` pasan, la construcción de M2 y M3 es correcta a nivel de shapes.
Los valores definitivos (accuracy, conteos sobre los pesos reales) se confirman al
ejecutar en Colab.


In [ ]:
# Verificación de arquitectura de M2 y M3 SIN descargar
# pesos preentrenados (weights=None) ni datos (tensores aleatorios).
# No requiere GPU: se ejecuta en CPU en segundos.
import torch
import torch.nn as nn
import torchvision

# Entrada aleatoria con forma [B, 3, 224, 224] (B=2). No se usa ninguna imagen real.
B = 2
x = torch.randn(B, 3, 224, 224)

# --- M2: VGG-16 con classifier = nn.Linear(25088, 2) y features congeladas ---
m2 = torchvision.models.vgg16(weights=None)
m2.classifier = nn.Linear(25088, 2)
for p in m2.features.parameters():
    p.requires_grad = False
m2.eval()
with torch.no_grad():
    out_m2 = m2(x)
assert out_m2.shape == (B, 2), f"M2: shape inesperado {tuple(out_m2.shape)}"
m2_entrenables = sum(p.numel() for p in m2.parameters() if p.requires_grad)
print('M2 (VGG-16) out.shape:', tuple(out_m2.shape))
print('M2 entrenables:', m2_entrenables)

# --- M3: ResNet-18 con fc determinando in_features por código y backbone congelado ---
m3 = torchvision.models.resnet18(weights=None)
# Congelamos todo el backbone primero...
for p in m3.parameters():
    p.requires_grad = False
# ...y reemplazamos fc determinando sus entradas POR CÓDIGO (no por texto).
m3.fc = nn.Linear(m3.fc.in_features, 2)
# nn.Linear nuevo nace con requires_grad=True; lo dejamos explícito por claridad.
for p in m3.fc.parameters():
    p.requires_grad = True
m3.eval()
with torch.no_grad():
    out_m3 = m3(x)
assert out_m3.shape == (B, 2), f"M3: shape inesperado {tuple(out_m3.shape)}"
m3_entrenables = sum(p.numel() for p in m3.parameters() if p.requires_grad)
print('M3 (ResNet-18) out.shape:', tuple(out_m3.shape))
print('M3 entrenables:', m3_entrenables)

print('\nVerificación OK: M2 y M3 construidos y verificados con weights=None y tensores aleatorios.')
